In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, Dict

# Input Models

class TariffStatus(Enum):
    OFF_PEAK = "OFF-PEAK"
    NORMAL = "NORMAL"
    PEAK = "PEAK"

@dataclass(frozen=True)
class CostContextRecord:
    """Module 5 input."""
    tariff_status: TariffStatus
    cost_severity_index: int  # 1-10

# Output Model

@dataclass(frozen=True)
class PeakMitigationRecord:
    """Peak mitigation record."""
    is_mitigation_active: bool
    mitigation_strategy: str        # Strategy
    peak_shaving_target_kw: float   # kW
    ups_safety_buffer: float        # %
    urgency_score: int              # 1-10

# Module Implementation

class EnerviaPeakOptimizer:
    """Peak Demand Optimizer."""

    def __init__(self, facility_peak_limit_kw: float = 1000.0):
        # Config
        self.FACILITY_PEAK_LIMIT = facility_peak_limit_kw
        self.UPS_CRITICAL_LOAD_THRESHOLD = 85.0  # Limit
        
    def evaluate_peak_demand(
        self, 
        facility_power: float, 
        ups_load: float, 
        cost_context: CostContextRecord
    ) -> PeakMitigationRecord:
        """Evaluate peak demand."""
        
        # UPS Safety Check
        ups_buffer_remaining = 100.0 - ups_load
        if ups_load > self.UPS_CRITICAL_LOAD_THRESHOLD:
            return PeakMitigationRecord(
                is_mitigation_active=False,
                mitigation_strategy="NONE_SAFETY_LOCKOUT",
                peak_shaving_target_kw=0.0,
                ups_safety_buffer=ups_buffer_remaining,
                urgency_score=0
            )

        # Identify opportunity
        is_peak_window = (cost_context.tariff_status == TariffStatus.PEAK)
        is_near_peak_limit = (facility_power > (self.FACILITY_PEAK_LIMIT * 0.9))
        
        should_mitigate = is_peak_window or is_near_peak_limit

        if not should_mitigate:
            return PeakMitigationRecord(
                is_mitigation_active=False,
                mitigation_strategy="STANDARD_OPERATIONS",
                peak_shaving_target_kw=0.0,
                ups_safety_buffer=ups_buffer_remaining,
                urgency_score=1
            )

        # Compute strategy
        strategy = "MODERATE_PEAK_SHAVE"
        if cost_context.cost_severity_index >= 8:
            strategy = "AGGRESSIVE_LOAD_REDUCTION"

        # Calc target
        target_kw = facility_power * 0.05 

        return PeakMitigationRecord(
            is_mitigation_active=True,
            mitigation_strategy=strategy,
            peak_shaving_target_kw=round(target_kw, 2),
            ups_safety_buffer=ups_buffer_remaining,
            urgency_score=cost_context.cost_severity_index
        )

# peak_engine = EnerviaPeakOptimizer(facility_peak_limit_kw=1500.0)
# record = peak_engine.evaluate_peak_demand(
#     facility_power=1420.0, 
#     ups_load=65.5, 
#     cost_context=cost_context_from_mod5
# )